# Kuramoto Handbook Workflow

Deterministic companion notebook for `examples/29_kuramoto_handbook_workflow.py`. It uses the public `scpn_quantum_control.kuramoto` facade for the Phase 5 handbook path: RK4 trajectory, frequency locking, stability spectrum, coherence clusters, critical coupling, and synchronising-coupling design.

In [ ]:
import numpy as np

import scpn_quantum_control.kuramoto as kuramoto

dt = 0.02
theta0 = np.array([0.0, 0.28, 0.62, 1.05, 1.52, 2.05], dtype=np.float64)
omega = np.array([-0.24, -0.10, -0.03, 0.05, 0.14, 0.22], dtype=np.float64)
coupling = np.array(
    [
        [0.0, 0.38, 0.12, 0.0, 0.0, 0.05],
        [0.38, 0.0, 0.34, 0.08, 0.0, 0.0],
        [0.12, 0.34, 0.0, 0.32, 0.08, 0.0],
        [0.0, 0.08, 0.32, 0.0, 0.35, 0.12],
        [0.0, 0.0, 0.08, 0.35, 0.0, 0.37],
        [0.05, 0.0, 0.0, 0.12, 0.37, 0.0],
    ],
    dtype=np.float64,
)

In [ ]:
trajectory = kuramoto.kuramoto_rk4_trajectory(theta0, omega, coupling, dt, 80)
frequency = kuramoto.frequency_order_diagnostics(trajectory, dt=dt, tolerance=0.08)
stability = kuramoto.stability_spectrum(trajectory[-1], coupling)
coherence = kuramoto.coherence_matrix(trajectory[-1])
clusters = kuramoto.cluster_partition(coherence, threshold=0.70)

{
    "tier": kuramoto.last_kuramoto_rk4_trajectory_tier_used(),
    "initial_order": float(kuramoto.order_parameter(theta0)),
    "final_order": float(kuramoto.order_parameter(trajectory[-1])),
    "frequency_index": float(frequency.synchronisation_index),
    "locked_fraction": float(frequency.locked_fraction),
    "spectral_gap": float(stability.spectral_gap),
    "cluster_count": int(clusters.count),
}

In [ ]:
designed = kuramoto.design_synchronising_coupling(
    theta0,
    omega,
    coupling,
    dt,
    40,
    max_iterations=6,
    learning_rate=1.5,
)
designed_trajectory = kuramoto.kuramoto_rk4_trajectory(theta0, omega, designed.coupling, dt, 80)

{
    "designed_final_order": float(kuramoto.order_parameter(designed_trajectory[-1])),
    "design_iterations": designed.iterations,
    "design_converged": designed.converged,
    "cost_history": tuple(float(value) for value in designed.cost_history),
}